In [ ]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


INPUT_FILE = "소재지정리.xlsx"      
OUTPUT_FILE = "토지이음_수집데이터.xlsx"
BASE_URL = "https://www.eum.go.kr/web/cp/cv/cvUpisDet.jsp"

WAIT = 10
TEST_N = None  


# =========================
# 1. 기본 함수
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().split())


def normalize_address(addr):
    if pd.isna(addr):
        return None

    addr = clean_text(addr)
    if addr == "":
        return None

    addr = re.sub(r"\s+", " ", addr)
    addr = re.sub(r"\s*-\s*", "-", addr)

    addr = addr.replace("울산시", "울산광역시")
    if addr.startswith("울산 "):
        addr = addr.replace("울산 ", "울산광역시 ", 1)

    if not addr.startswith("울산광역시"):
        addr = "울산광역시 " + addr

    compact = addr.replace(" ", "")
    if re.fullmatch(r"울산광역시\d+(?:-\d+)?", compact):
        return None

    return addr


# =========================
# 2. 입력 데이터 로드
# =========================
def load_addresses():
    df = pd.read_excel(INPUT_FILE)

    df.columns = [clean_text(c) for c in df.columns]

    if "소재지" not in df.columns:
        raise KeyError("입력 파일에 '소재지' 컬럼이 없습니다.")

    df = df.copy()
    df["소재지"] = df["소재지"].apply(normalize_address)
    df = df[df["소재지"].notna()].copy()
    df = df.drop_duplicates(subset=["소재지"]).reset_index(drop=True)

    if TEST_N:
        df = df.iloc[:TEST_N].copy()

    return df


# =========================
# 3. 브라우저
# =========================
def setup_driver():
    options = webdriver.EdgeOptions()
    options.add_argument("--start-maximized")
    return webdriver.Edge(options=options)


def open_page(driver):
    driver.get(BASE_URL)
    WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located((By.XPATH, "//input[contains(@placeholder,'지번')]"))
    )


# =========================
# 4. 검색
# =========================
def search_address(driver, address):
    box = WebDriverWait(driver, WAIT).until(
        EC.element_to_be_clickable((By.XPATH, "//input[contains(@placeholder,'지번')]"))
    )
    box.clear()
    box.send_keys(address)
    time.sleep(0.5)

    box.send_keys(Keys.ENTER)
    time.sleep(0.7)

    btn = WebDriverWait(driver, WAIT).until(
        EC.element_to_be_clickable(
            (By.XPATH, "//*[normalize-space(text())='열람']/ancestor::a | //*[normalize-space(text())='열람']/ancestor::button")
        )
    )
    driver.execute_script("arguments[0].click();", btn)
    time.sleep(1.2)


# =========================
# 5. 팝업 / 모달 체크
# =========================
def has_error_popup(driver):
    try:
        info = driver.execute_script("""
            const el = document.querySelector('.layer_pop.error_layer');
            if (!el) return false;
            const style = window.getComputedStyle(el);
            const text = el.innerText || '';
            return style.display !== 'none' && text.length > 0;
        """)
        return bool(info)
    except:
        return False


def close_error_popup_if_exists(driver):
    xpaths = [
        "//a[@class='layer_close' and contains(@href, \"closeLayer('error_layer')\")]",
        "//div[contains(@class,'error_layer')]//a[contains(@class,'layer_close')]",
        "//div[contains(@class,'error_layer')]//button"
    ]

    for xp in xpaths:
        try:
            btn = WebDriverWait(driver, 1.5).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.5)
            return True
        except:
            pass
    return False


def open_land_history(driver):
    btn = WebDriverWait(driver, WAIT).until(
        EC.element_to_be_clickable((By.XPATH, "//*[contains(text(),'토지이력') and contains(text(),'특성')]"))
    )
    driver.execute_script("arguments[0].click();", btn)
    time.sleep(0.8)


def close_modal(driver):
    xpaths = [
        "//button[contains(@class,'ui-dialog-titlebar-close')]",
        "//span[contains(@class,'ui-icon-closethick')]/parent::*",
        "//div[contains(@class,'ui-dialog')]//button[@type='button']"
    ]

    for xp in xpaths:
        try:
            btn = WebDriverWait(driver, 1).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.3)
            return
        except:
            pass


# =========================
# 6. 토지이력 파싱
# =========================
def parse_land_history(driver):
    WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located((By.XPATH, "//*[contains(text(),'관련 정보 보기')]"))
    )

    header_row = WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located((
            By.XPATH,
            "//tr[.//td[contains(normalize-space(.), '토지이동')] and .//*[contains(normalize-space(.), '지목')] and .//*[contains(normalize-space(.), '면적')] and .//*[contains(normalize-space(.), '토지이동일')]]"
        ))
    )

    rows = header_row.find_elements(By.XPATH, "following-sibling::tr")
    records = []

    for row in rows:
        texts = [clean_text(td.text) for td in row.find_elements(By.XPATH, "./td")]

        if len(texts) < 4:
            continue

        land_type = texts[0]
        area = texts[1]
        reason = texts[2]          # ⭐ 추가
        move_date_raw = texts[-1]

        dt = pd.to_datetime(move_date_raw, errors="coerce")
        if pd.notna(dt):
            records.append({
                "지목": land_type,
                "면적": area,
                "토지이동사유": reason,   # ⭐ 추가
                "토지이동일": dt
            })

    if not records:
        raise RuntimeError("토지이동 데이터 추출 실패")

    work = pd.DataFrame(records)
    latest = work.sort_values("토지이동일", ascending=False).iloc[0]

    return {
        "지목": clean_text(latest["지목"]),
        "면적": clean_text(latest["면적"]),
        "토지이동사유": clean_text(latest["토지이동사유"]),  # ⭐ 추가
        "토지이동일": latest["토지이동일"].strftime("%Y-%m-%d")
    }


# =========================
# 7. 한 건 처리
# =========================
def process_one(driver, address):
    search_address(driver, address)

    popup_found = has_error_popup(driver)
    popup_closed = False

    if popup_found:
        popup_closed = close_error_popup_if_exists(driver)

    open_land_history(driver)
    result = parse_land_history(driver)
    close_modal(driver)

    return {
        "지목": result["지목"],
        "면적": result["면적"],
        "토지이동사유": result["토지이동사유"],   # ⭐ 추가
        "토지이동일": result["토지이동일"],
        "상태": "오류팝업후조회" if popup_found or popup_closed else "정상"
    }


# =========================
# 8. 실행
# =========================
def main():
    df = load_addresses()
    print("대상 건수:", len(df))

    results = []
    driver = None

    try:
        driver = setup_driver()
        open_page(driver)

        for i, row in df.iterrows():
            address = row["소재지"]
            print(f"[{i+1}/{len(df)}] {address}")

            try:
                record = process_one(driver, address)

            except Exception as e:
                record = {
                    "지목": "",
                    "면적": "",
                    "토지이동사유": "",  
                    "토지이동일": "",
                    "상태": "확인필요"
                }

                try:
                    close_modal(driver)
                except:
                    pass

                try:
                    close_error_popup_if_exists(driver)
                except:
                    pass

                try:
                    open_page(driver)
                except:
                    pass

            results.append(record)

        result_df = pd.DataFrame(results)
        final_df = pd.concat([df.reset_index(drop=True), result_df], axis=1)
        final_df.to_excel(OUTPUT_FILE, index=False)

        print("완료:", OUTPUT_FILE)

    finally:
        if driver is not None:
            try:
                driver.quit()
            except:
                pass


if __name__ == "__main__":
    main()

대상 건수: 393
[1/393] 울산광역시 남구 고사동 1-2
[2/393] 울산광역시 남구 고사동 110-26
[3/393] 울산광역시 남구 고사동 110-27
[4/393] 울산광역시 남구 고사동 110-28
[5/393] 울산광역시 남구 고사동 110-29
[6/393] 울산광역시 남구 고사동 129-11
[7/393] 울산광역시 남구 고사동 17-2
[8/393] 울산광역시 남구 고사동 2-9
[9/393] 울산광역시 남구 남화동 350
[10/393] 울산광역시 남구 남화동 351
[11/393] 울산광역시 남구 남화동 352
[12/393] 울산광역시 남구 남화동 53-3
[13/393] 울산광역시 남구 남화동 53-4
[14/393] 울산광역시 남구 매암동 1-10
[15/393] 울산광역시 남구 매암동 1-12
[16/393] 울산광역시 남구 매암동 1-13
[17/393] 울산광역시 남구 매암동 1-17
[18/393] 울산광역시 남구 매암동 1-3
[19/393] 울산광역시 남구 매암동 1-5
[20/393] 울산광역시 남구 매암동 139-1
[21/393] 울산광역시 남구 매암동 139-10
[22/393] 울산광역시 남구 매암동 139-12
[23/393] 울산광역시 남구 매암동 139-14
[24/393] 울산광역시 남구 매암동 139-17
[25/393] 울산광역시 남구 매암동 139-29
[26/393] 울산광역시 남구 매암동 139-30
[27/393] 울산광역시 남구 매암동 139-39
[28/393] 울산광역시 남구 매암동 139-40
[29/393] 울산광역시 남구 매암동 139-41
[30/393] 울산광역시 남구 매암동 139-43
[31/393] 울산광역시 남구 매암동 139-45
[32/393] 울산광역시 남구 매암동 139-47
[33/393] 울산광역시 남구 매암동 139-48
[34/393] 울산광역시 남구 매암동 139-50
[35/393] 울산광역시 남구 매암동 139-52
[36/393] 울산광역시 남구 매